# IoT-NetGuard — Model 3: Decision Trees
**Project:** Anomaly Detection for IoT Cybersecurity  
**Dataset:** `iot23_combined.csv` (output of `01_Data_Preprocessing.ipynb`)  
**Algorithm:** Decision Tree Classifier (Gini impurity) — interpretable rule-based model

## 1. Imports

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
print('All imports successful.')

## 2. Load Dataset

In [ ]:
FILEPATH = "iot23_combined.csv"
FEATURES = [
    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
    'proto_icmp', 'proto_tcp', 'proto_udp',
    'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0',
    'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1',
    'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR'
]
TARGET = 'label'

df = pd.read_csv(FILEPATH)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print(f'Dataset shape : {df.shape}')
print(f'Label distribution:\n{df[TARGET].value_counts().to_string()}')

## 3. Feature Selection & Train/Test Split

In [ ]:
X = df[FEATURES]
Y = df[TARGET]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=10, stratify=Y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing  samples : {X_test.shape[0]:,}')

## 4. Train Decision Tree

In [ ]:
print('Training Decision Tree...')
start = time.time()

dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=None,       # Fully grown tree
    min_samples_split=2,
    random_state=42
)
dt.fit(X_train, Y_train)

elapsed = time.time() - start
print(f'Training complete in {elapsed:.2f} seconds')
print(f'Tree depth : {dt.get_depth()}')
print(f'Leaf nodes : {dt.get_n_leaves():,}')

## 5. Evaluation

In [ ]:
Y_pred   = dt.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)

print(f'Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Time cost : {elapsed:.2f} seconds')
print()
print('Classification Report:')
print(classification_report(Y_test, Y_pred))

## 6. Feature Importance

In [ ]:
importances = pd.Series(dt.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
importances.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Decision Tree — Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance (Gini)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/dt_feature_importance.png', dpi=150)
plt.show()
print('Saved → outputs/dt_feature_importance.png')

## 7. Confusion Matrix

In [ ]:
labels = sorted(df[TARGET].unique())
cm = confusion_matrix(Y_test, Y_pred, labels=labels)

plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels)
plt.title('Decision Tree — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/dt_confusion_matrix.png', dpi=150)
plt.show()
print('Saved → outputs/dt_confusion_matrix.png')

## 8. Summary

In [ ]:
print('=' * 48)
print('  IoT-NetGuard — Decision Tree Summary')
print('=' * 48)
print(f'  Model      : DecisionTreeClassifier (Gini)')
print(f'  Features   : {len(FEATURES)}')
print(f'  Train/Test : 80% / 20%')
print(f'  Tree depth : {dt.get_depth()}')
print(f'  Accuracy   : {accuracy*100:.2f}%')
print(f'  Time       : {elapsed:.2f}s')
print('=' * 48)